In [12]:
import pandas as pd
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,f1_score

In [13]:
df = pl.read_parquet("../data/03_processed/baseline_features_train.parquet").to_pandas()
df = df.dropna(subset='IncidentGrade')
label_map = {
    "FalsePositive" : 0,
    'BenignPositive' : 1,
    'TruePositive' : 2
}
df['target'] = df['IncidentGrade'].map(label_map)


X = df.drop(columns=['IncidentGrade','start_time','end_time','target','OrgId','IncidentId'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training set shape: (358277, 31)
Validation set shape: (89570, 31)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [14]:
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
print("Training the Random Forest Model ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n--- Classification Report ---")
target_labels = ['False Positive (0)','Benign Positive (1)','True Positive (2)']
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.81      0.60      0.69     26978
Benign Positive (1)       0.67      0.87      0.76     43509
  True Positive (2)       0.61      0.39      0.48     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.62      0.64     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6430


## Baseline model on engineered features

In [19]:
df_improved = pl.read_parquet("../data/03_processed/engineered_features_train.parquet").to_pandas()
df_improved = df_improved.dropna(subset='IncidentGrade')
df_improved['target'] = df_improved['IncidentGrade'].map(label_map)

X_improved = df_improved.drop(columns=['IncidentGrade','target','OrgId','IncidentId'])
y_improved = df_improved['target']

X_train_improved,X_test_improved,y_train_improved,y_test_improved = train_test_split(X_improved,y_improved,test_size=0.2,random_state=42,stratify=y_improved)

print(f"Training set shape: {X_train_improved.shape}")
print(f"Validation set shape: {X_test_improved.shape}")
print(f"Target distribution:\n{y_train_improved.value_counts(normalize=True) * 100}")

Training set shape: (358277, 37)
Validation set shape: (89570, 37)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [20]:
rf_baseline_improved = RandomForestClassifier(
    n_estimators = 50,
    max_depth=10,
    random_state=42,
    n_jobs = -1
)

print("Training the Random Forest Model ...")
rf_baseline_improved.fit(X_train_improved,y_train_improved)

print("Predicting on validation set ...")
y_pred_improved = rf_baseline_improved.predict(X_test_improved)

print("\n--- Classification Report ---")
print(classification_report(y_test_improved,y_pred_improved,target_names=target_labels))

macro_f1_improved = f1_score(y_test_improved,y_pred_improved,average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1_improved:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.81      0.61      0.70     26978
Benign Positive (1)       0.67      0.87      0.76     43509
  True Positive (2)       0.62      0.40      0.49     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.63      0.65     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6459


In [21]:
importance = pd.Series(rf_baseline_improved.feature_importances_,index=X_train_improved.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Top 15 Most Important Features:
evidence_per_second             0.089871
total_evidence_count            0.079487
unique_state_count              0.070856
unique_entitytype_count         0.067229
unique_countrycode_count        0.065187
unique_city_count               0.063227
is_multinational                0.061812
unique_sha256_count             0.038290
unique_filename_count           0.035781
unique_url_count                0.034859
ips_per_device                  0.032254
unique_accountobjectid_count    0.029955
unique_accountupn_count         0.029347
unique_applicationname_count    0.027688
unique_applicationid_count      0.027437
dtype: float64
